In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

### a. Normalize features and perform one-hot encode labels

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.3, random_state=42)

### b & c. Implement feedforward NN with 1 hidden layer & backpropagation from scratch

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def cross_entropy(y_true, y_pred):
    return -np.sum(y_true * np.log(y_pred + 1e-9)) / y_true.shape[0]

input_size = X_train.shape[1]
hidden_size = 10
output_size = y_train.shape[1]
learning_rate = 0.1
epochs = 1000

W1 = np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

losses = []

for epoch in range(epochs):
    z1 = X_train @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    
    loss = cross_entropy(y_train, a2)
    losses.append(loss)
    
    dz2 = a2 - y_train
    dW2 = a1.T @ dz2 / X_train.shape[0]
    db2 = np.sum(dz2, axis=0, keepdims=True) / X_train.shape[0]
    
    dz1 = dz2 @ W2.T * sigmoid_derivative(a1)
    dW1 = X_train.T @ dz1 / X_train.shape[0]
    db1 = np.sum(dz1, axis=0, keepdims=True) / X_train.shape[0]
    
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

### d. Report accuracy, plot loss, and confusion matrix

In [ ]:
train_pred = np.argmax(sigmoid(sigmoid(X_train @ W1 + b1) @ W2 + b2), axis=1)
train_true = np.argmax(y_train, axis=1)
print(f"Training Accuracy: {np.mean(train_pred == train_true) * 100:.2f}%")

z1 = X_test @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2
a2 = sigmoid(z2)
test_pred = np.argmax(a2, axis=1)
test_true = np.argmax(y_test, axis=1)
print(f"Test Accuracy: {np.mean(test_pred == test_true) * 100:.2f}%")

plt.plot(losses)
plt.title('Loss vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Cross-Entropy Loss')
plt.show()

cm = confusion_matrix(test_true, test_pred)
sns.heatmap(cm, annot=True, cmap='Blues')
plt.title('Confusion Matrix on Test Set')
plt.show()